# SD1.5 LoRA 训练（Kaggle 免费 T4）

本机只有 4 GB 显存，LoRA 训练放 Kaggle 的免费 T4（16 GB）上跑。

## 使用步骤

1. Kaggle → Create → New Notebook
2. 右侧 Settings：
   - **Accelerator: GPU T4 x2**
   - **Internet: On**
3. 数据集（15-30 张图 + 同名 .txt 标签）打包成 zip，上传为 Kaggle Dataset，名字记作 `lora-data`
4. 从上到下依次运行 cell
5. 训完在 `/kaggle/working/lora_out` 里下载 `.safetensors`（几十 MB）

免费额度：每周 30 小时 GPU。训一个 LoRA 约 30-60 分钟。


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda 可用', torch.cuda.is_available())
print('设备', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')


## 1. 装 sd-scripts（kohya）

In [ ]:
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q -r requirements.txt
!pip install -q bitsandbytes xformers
!accelerate config default
print('环境装好')


## 2. 准备底模

下载 SD1.5 官方权重（约 4.3 GB）。慢的话可以自己上传到 Kaggle Dataset 再引用路径。

In [ ]:
import os
os.makedirs('/kaggle/working/models', exist_ok=True)
!wget -q -O /kaggle/working/models/sd15.safetensors https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
!ls -lh /kaggle/working/models/


## 3. 准备数据集

解压到 `/kaggle/working/train_data`，结构：

```
train_data/
  img001.png
  img001.txt   ← 标签，例如 "a photo of sks style, product on table"
  img002.png
  img002.txt
```

把下面的 `lora-data` 换成你实际的 dataset 名字。

In [ ]:
!mkdir -p /kaggle/working/train_data
!cp /kaggle/input/lora-data/*.zip /kaggle/working/ 2>/dev/null || true
!cd /kaggle/working && for z in *.zip; do unzip -o -q "$z" -d train_data; done
!ls /kaggle/working/train_data | head -20


## 4. 开始训练

In [ ]:
%cd /kaggle/working/sd-scripts

!accelerate launch --num_cpu_threads_per_process 2 train_network.py \
  --pretrained_model_name_or_path=/kaggle/working/models/sd15.safetensors \
  --train_data_dir=/kaggle/working/train_data \
  --output_dir=/kaggle/working/lora_out \
  --output_name=style_demo \
  --network_module=networks.lora \
  --network_dim=32 --network_alpha=16 \
  --resolution=512,512 \
  --train_batch_size=2 \
  --max_train_epochs=10 \
  --learning_rate=1e-4 --text_encoder_lr=5e-5 --unet_lr=1e-4 \
  --optimizer_type=AdamW8bit \
  --lr_scheduler=cosine --lr_warmup_steps=50 \
  --mixed_precision=fp16 --save_precision=fp16 \
  --save_model_as=safetensors \
  --save_every_n_epochs=2 \
  --cache_latents --gradient_checkpointing --xformers \
  --seed=42 \
  --caption_extension=.txt


## 5. 看结果

训练日志里会打印 `avr_loss=...`，正常从 0.2 左右降到 0.1 上下。

**要留下的证据（面试用）**：

1. loss 下降的截图
2. `style_demo.safetensors` 的文件列表
3. 同一个 prompt 下，base 模型 vs LoRA 模型的对比图（把 LoRA 下载到本机 `ComfyUI/models/loras/` 后跑）

In [ ]:
!ls -lh /kaggle/working/lora_out/


## 常见问题

| 现象 | 原因 | 处理 |
|---|---|---|
| `CUDA out of memory` | 参数太大 | `train_batch_size` 降 1，或 `--network_dim=16` |
| `No module named xformers` | 没装上 | 重跑安装 cell，必要时重启 kernel |
| 训练完效果差 | 数据少 / 标签乱 | 至少 15 张，标签写清「主体 + 风格」 |
| 底模下载 403 | HuggingFace 要登录 | 自己上传底模到 Kaggle Dataset |
